In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# import cv2
import scipy as sp
from skimage import io
from skimage.color import rgb2hsv, hsv2rgb

gamma = 2.2
t = 1/2048

Z_min = 0.02
Z_max = 0.95

Npictures = 16

t_list = np.array([t*2**k for k in range(Npictures)])

In [ ]:
image_scale = 200

In [ ]:
def prepare_image_stack(image_stack):
    # by default, image_stack is a list of N numpy arrays, shapes (H,W,3).
    # This function reshapes them to a single numpy array of (N, H*W*3)
    reshaped_stack = np.array([img.flatten() for img in image_stack])
    return reshaped_stack

image_stack = [io.imread(f"data/door_stack/exposure{i}.jpg")[::image_scale,::image_scale,:] for i in range(1, 17)]

plt.imshow(image_stack[11])

image_stack = prepare_image_stack(image_stack)

In [ ]:
image_stack.shape

In [ ]:
image_stack[15].max()

In [ ]:
def construct_least_squares_matrices(image_stack, 
                                     weight_function: callable,
                                     exposure_times: np.ndarray,
                                     lambda_smooth: float = 1000.0,
                                     grey_scales: int = 256):
    
    """ 

    Convention:
    image_stack: shape (n_images, n_pixels * n_colors) since we do not separate colors here
    exposure_times: shape (n_images, )

    We construct the matrices Q and vector c such that we can write the expression 
    v.T @ Q @ v + c.T @ v

    Afterwards we obtain A and b such that we can solve the equation
    A @ v = b
    """
    
    assert image_stack.shape[0] == exposure_times.shape[0], "Number of images and exposure times must match"

    Npixels = image_stack.shape[1]
    Nimages = image_stack.shape[0]

    Q = np.zeros((grey_scales + Npixels, grey_scales + Npixels))
    c = np.zeros(grey_scales + Npixels)

    # Data term
    for i in range(Npixels):
        for k in range(Nimages):
            Iik = image_stack[k, i] # What they call I_ij^k in the exercise
            w_squared = weight_function(float(Iik) / 255.)**2

            Q[Iik, Iik] += w_squared
            Q[Iik, grey_scales + i] -= w_squared
            Q[grey_scales + i, Iik] -= w_squared
            Q[grey_scales + i, grey_scales + i] += w_squared

            c[Iik] -= 2 * w_squared * np.log(exposure_times[k]) 
            c[grey_scales + i] += 2 * w_squared * np.log(exposure_times[k])
    
    # Smoothness term
    for z in range(1, grey_scales - 1):
        w_squared = weight_function(float(z) / 255.)**2

        Q[z-1, z-1] += lambda_smooth * w_squared
        Q[z, z] += 2 * lambda_smooth * w_squared
        Q[z+1, z+1] += lambda_smooth * w_squared

        Q[z-1, z] -= lambda_smooth * w_squared
        Q[z, z-1] -= lambda_smooth * w_squared

        Q[z, z+1] -= lambda_smooth * w_squared
        Q[z+1, z] -= lambda_smooth * w_squared

    alpha = 1e6  # strong but finite
    Q[128, 128] += alpha

    # obtain A and b
    w, U = np.linalg.eigh(Q)
    w = np.clip(w, 0, None)        # remove tiny negative values
    A = np.diag(np.sqrt(w)) @ U.T    
    b, *_ = np.linalg.lstsq(A.T, -c / 2, rcond=None)

    # solve for A @ v = b
    v, *_ = np.linalg.lstsq(A, b, rcond=None)

    return v

def g_from_v(v: np.ndarray,
             grey_scales: int = 256) -> np.ndarray:
    """ Extract g from v """
    global Z_min, Z_max
    min_pixel_value = int(np.ceil(Z_min * 255))
    max_pixel_value = int(np.floor(Z_max * 255) + 1)

    g = np.zeros(grey_scales)
    g[min_pixel_value:max_pixel_value] = v[min_pixel_value:max_pixel_value]

    g[:min_pixel_value] = v[min_pixel_value]
    g[max_pixel_value:] = v[max_pixel_value]
    
    return g

def linearize_image(image: np.ndarray,
                     g: np.ndarray) -> np.ndarray:
    """ Linearize a single image using g """
    H, W, C = image.shape
    linearized_image = np.zeros((H, W, C), dtype=np.float32)

    for c in range(C):
        for i in range(H):
            for j in range(W):
                pixel_value = image[i, j, c]
                linearized_image[i, j, c] = np.exp(g[pixel_value])

    return linearized_image

In [ ]:
def w_uniform(z, tk=1):
  return np.logical_and(Z_min <= z, z<= Z_max)

def w_tent(z, tk=1):
  w = np.minimum(z, np.ones_like(z) - z)
  return w * w_uniform(z)

def w_Gaussian(z, tk=1):
  w = np.exp(-16 * (z - 0.5) ** 2)
  return w * w_uniform(z)

def w_photon(z, tk):
  return tk * w_uniform(z)

valid_w = ['uniform', 'tent', 'Gaussian', 'photon']
def w(z, w_type='uniform', tk=1):
  if w_type == 'uniform':
    return w_uniform(z, tk)
  elif w_type == 'tent':
    return w_tent(z, tk)
  elif w_type == 'Gaussian':
    return w_Gaussian(z, tk)
  elif w_type == 'photon':
    return w_photon(z, tk)
  else:
    raise ValueError('Invalid weight function')

In [ ]:
Z_max * 255


In [ ]:


for lambda_smooth in [1e1, 1e2, 1e3, 1e4, 1e5, 1e6]:
    v = construct_least_squares_matrices(image_stack, w_tent, t_list, lambda_smooth=lambda_smooth)
    plt.plot(v[6:242], label=f'lambda = {lambda_smooth}')

plt.xlabel('Pixel value')
plt.ylabel('Log exposure')
plt.legend()
plt.show()

In [ ]:
g = g_from_v(construct_least_squares_matrices(image_stack, w_tent, t_list, lambda_smooth=100))

In [ ]:
plt.plot(g)
plt.xlabel('Pixel value')
plt.ylabel('Log exposure')
plt.show()

In [ ]:
image_stack_linear = [linearize_image(io.imread(f"data/door_stack/exposure{i}.jpg")[::image_scale,::image_scale,:], g) for i in range(1, 17)]
plt.imshow(image_stack_linear[11])

In [ ]:
# now do the HDR pipeline 